In [2]:
import numpy as np 
import pandas as pd

In [31]:
df = pd.read_csv(r"C:\Users\HP\Downloads\pharmaly_products_rag_ready.csv")


In [ ]:
import pandas as pd
import chromadb
import hashlib
from pathlib import Path
from sentence_transformers import SentenceTransformer

C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LOAD THE EMBEDDING MODEL

In [22]:
print("\nLoading embedding model...")

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded!")


Loading embedding model...


C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights

Embedding model loaded!


In [37]:
def get_embeddings(texts):
    """
    Convert a list of texts into embeddings.
    Returns a list of numerical vectors.
    """
    if not texts:
        return []

    embeddings = model.encode(
        texts,
        batch_size=32,
        show_progress_bar=False,
        normalize_embeddings=True
    )

    return embeddings.tolist()
def get_embedding(text):
    """
    Convert one text into one embedding.
    Useful later for Member 3 when embedding user questions.
    """

    return get_embeddings([text])[0]


if __name__ == "__main__":
    example = "Vitamin C supplement for immunity"

    vector = get_embedding(example)

    print("Embedding generated successfully!")
    print("Vector dimensions:", len(vector))
    print("First 5 values:", vector[:5])

Embedding generated successfully!
Vector dimensions: 384
First 5 values: [-0.07421910017728806, 0.05656564608216286, -0.0670352503657341, 0.012451119720935822, 0.06838870793581009]


Create and manage our persistent ChromaDB database.

In [45]:
import chromadb

# 1. Define database location
DB_PATH = "./chroma_db"

# 2. Create a persistent ChromaDB client
client = chromadb.PersistentClient(
    path=DB_PATH
)

# 3. Create or access our collection
collection = client.get_or_create_collection(
    name="pharmaly_products",
    metadata={"hnsw:space": "cosine"}
)


# 4. Function to store products
def add_documents(ids, documents, embeddings, metadatas):

    collection.upsert(
        ids=ids,
        documents=documents,
        embeddings=embeddings,
        metadatas=metadatas
    )


# 5. Function to count stored products
def get_collection_count():

    return collection.count()


# 6. Display database information
print("Connected to ChromaDB!")
print("Collection:", collection.name)
print("Number of stored products:", get_collection_count())

Connected to ChromaDB!
Collection: pharmaly_products
Number of stored products: 0


embadding of documents

In [50]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

embeddings = model.encode(
    documents,
    show_progress_bar=True
)

print("Embeddings generated:", len(embeddings))

Batches: 100%|██████████| 286/286 [13:23<00:00,  2.81s/it]

Embeddings generated: 9148


In [53]:
ids = [str(i) for i in range(len(documents))]

In [58]:
print(df.columns.tolist())
df.head(3)

['product_name', 'brand', 'categories', 'original_price', 'current_price', 'description', 'product_url']


,product_name,brand,categories,original_price,current_price,description,product_url
0,Cheks Postbiotic Balance Bt/45,Cheks,Sante,NaN,2700.0,Le Cheks Postbiotic Balance est un complément ...,https://www.pharmaly-dz.com/produit/cheks-post...
1,Acm Novophane Complément Alimentaire Ongles/ch...,Laboratoires ACM,Sante,NaN,6500.0,La crème apaisante ACM Sédacalm 120ml est spéc...,https://www.pharmaly-dz.com/produit/acm-novoph...
2,Granions Biotine Vitamine B8 46g,Granions,Sante,NaN,4800.0,Granions Biotine Vitamine B8 est un complément...,https://www.pharmaly-dz.com/produit/granions-b...


metadata

In [59]:
import pandas as pd

metadatas = []

for _, row in df.iterrows():

    metadata = {
        "product_name": str(row["product_name"]),
        "brand": str(row["brand"]) if pd.notna(row["brand"]) else "Not specified",
        "category": str(row["categories"]) if pd.notna(row["categories"]) else "Not specified",
        "product_url": str(row["product_url"]) if pd.notna(row["product_url"]) else ""
    }

    if pd.notna(row["current_price"]):
        metadata["current_price"] = float(row["current_price"])

    if pd.notna(row["original_price"]):
        metadata["original_price"] = float(row["original_price"])

    metadatas.append(metadata)

print("Number of metadata:", len(metadatas))

Number of metadata: 9148


Stocker dans chroma database

In [60]:
ids = [str(i) for i in range(len(documents))]

batch_size = 1000

for start in range(0, len(documents), batch_size):

    end = min(start + batch_size, len(documents))

    collection.upsert(
        ids=ids[start:end],
        documents=documents[start:end],
        embeddings=embeddings[start:end].tolist(),
        metadatas=metadatas[start:end]
    )

    print(f"Stored: {end}/{len(documents)}")

print("Total products:", collection.count())

Stored: 1000/9148
Stored: 2000/9148
Stored: 3000/9148
Stored: 4000/9148
Stored: 5000/9148
Stored: 6000/9148
Stored: 7000/9148
Stored: 8000/9148
Stored: 9000/9148
Stored: 9148/9148
Total products: 9148


In [61]:
results = collection.get(
    ids=["0", "1"],
    include=["documents", "metadatas"]
)

print("Document:")
print(results["documents"][0])

print("\nMetadata:")
print(results["metadatas"][0])

Document:
Product Name: Cheks Postbiotic Balance Bt/45
Brand: Cheks
Category: Sante
Current Price: 2700 DA

Description:
Le Cheks Postbiotic Balance est un complément alimentaire en gummies conçu pour soutenir l’équilibre du microbiote et le confort digestif. Sa formule associe un lysat postbiotique de Lactobacillus plantarum, du butyrate de calcium, de l’inuline ainsi que des extraits de gingembre et de menthe. Cette synergie accompagne la digestion, la régularité du transit et aide à limiter les sensations de ballonnement après les repas. Le Cheks Postbiotic Balance contient également du zinc et de la vitamine D3, qui contribuent au fonctionnement normal du système immunitaire. Ses gummies vegan sont sans sucre ajouté, sans gélatine et sans gluten.

Metadata:
{'product_url': 'https://www.pharmaly-dz.com/produit/cheks-postbiotic-balance-bt-45/', 'brand': 'Cheks', 'category': 'Sante', 'current_price': 2700.0, 'product_name': 'Cheks Postbiotic Balance Bt/45'}


In [48]:
print("Documents:", len(documents))


Documents: 9148


In [42]:
ids = [str(i) for i in range(len(documents))]

In [44]:
# Generate unique IDs
ids = [str(i) for i in range(len(documents))]

# Store products in ChromaDB
add_documents(
    ids=ids,
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas
)

print("Products successfully stored in ChromaDB!")
print("Total products:", get_collection_count())

NameError: name 'embeddings' is not defined

In [40]:
def create_product_document(row):

    document = f"""
    Product name: {row['product_name']}
    Brand: {row['brand']}
    Category: {row['categories']}
    Description: {row['description']}
    """

    return document.strip()